In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

# Single neuron

![experiment schematic](single_cell_recording.png "Single cell recording")
![simplified system](singe_cell_sim.svg)

This simplified model of a neuron is mathematically expressed with a first-order ordinary differential equation:

$$
\tau \frac{d r(t)}{dt} = -r(t) + I(t)
$$

which we can discretize in time (in steps $dt$) in order to simulate numerically using the Euler method:

$\tau \frac{r(t+dt) - r(t)}{dt} = -r(t) + I(t) \; \; \; $ so that:  $\; \; \; r(t+dt) = r(t) + \frac{dt}{\tau} \left[ -r(t) + I(t) \right]$

Here is a piece of code that simulates how a neuron with membrane time constant $\tau$ integrates a time varying input and generates a rate output $r(t)$

In [ ]:
dt = 0.001 # time step, in s
time = np.arange(0, 5, dt) # step times of the whole simulation, in s
Nt = len(time) # total number of steps in simulation

T = 0.5 # period of input current, in s

You can define different possible input currents. Let's start with a constant input current or a sinusoidal input current:

In [ ]:
input = 10*np.ones((Nt,))  # constant input current
# input = 1 + np.sin(2*np.pi*time/T) # sinusoidal input current

Now run the simulation (two lines of code!) and then plot the results:

In [ ]:
rate = np.ones((Nt,)) # initialize firing rate at 1

tau = 0.02 # membrane time constant, in s

# simulation loop, through all time steps
for i in range(Nt-1):
    rate[i+1] = rate[i] + dt/tau*(-rate[i] + input[i])

# plots
fig, ax = plt.subplots(nrows=2)
ax[0].plot(time, input, 'k')
ax[0].set_title('Input')
ax[0].set_ylabel('current')
ax[1].plot(time, rate, 'r')
ax[1].set_title('Output')
ax[1].set_xlabel('time (s)')
ax[1].set_ylabel('rate')

Now copy the code above to see how a neuron with a much longer time constant (e.g. $\tau = 1 s$) would integrate this same input:

Now compare again the two neurons, one with short time constant ($\tau = 0.02 s$) and one with very long time constant ($\tau = 10 s$), but now have them respond to an input representing a current pulse, as per the code below:

In [ ]:
input = np.ones((Nt,))
pulsesat = int(Nt/6)
input[pulsesat] = 3


What do you see?

Now do the same thing but for an input consisting in a train of successive current pulses:

In [ ]:
input = np.ones((Nt,))
pulsesat = int(Nt/6)*np.array([1,2,3,4,5])
input[pulsesat] = 3

This is quite exciting: a single neuron can do things such as memory, or stimulus counting (integration)... but, what is a realistic membrane time constant in the brain? 20ms!

How could the brain do memory or stimulus integration if neurons have such fast time constant? Networks! Here is the simplest possible network:

# Neuron with autapse

![Autapse](autapse.svg "Autapse")

Now this simplified network model of a neuron is mathematically expressed with this first-order ordinary differential equation:

$$
\tau \frac{d r(t)}{dt} = -r(t) + J r(t) + I(t)
$$

where $J$ is the strength of the self-coupling of the neuron with itself (autapse). We again discretize this equation in time (in steps $dt$) using the Euler method to get:

$r(t+dt) = r(t) + \frac{dt}{\tau} \left[ (J - 1) r(t) + I(t) \right]$

Now try simulating this equation for various values of $J$ (restricted to be non-negative, $J \ge 0$) when the neuron has a short membrane time constant (keep fixed $\tau = 0.02s$):

In [ ]:
input = np.ones((Nt,))
pulsesat = int(Nt/6)
input[pulsesat] = 3

rate = np.ones((Nt,))

tau = 0.02
J = 0.99

input = input*np.abs(1-J)
for i in range(Nt-1):
    rate[i+1] = rate[i] + dt/tau*((J - 1)*rate[i] + input[i])

fig, ax = plt.subplots(nrows=2)
ax[0].plot(time, input, 'k')
ax[0].set_title('Input')
ax[0].set_ylabel('current')
ax[1].plot(time, rate, 'r')
ax[1].set_title('Output')
ax[1].set_xlabel('time (s)')
ax[1].set_ylabel('rate')

In what conditions does this network show perfect memory (i.e. it has a different rate depending on whether a stimulus was presented in the past or not)?

What happens if you set $J>1$?

So, through recurrent connections, neurons with short membrane time constants can achieve memory and integration capabilities as if they had long time constants. This is one of the fundamental principles of network function in attractor networks.

However, as you have seen, this architecture alone has some fundamental issues:
1) fine-tuning of memory ($J=1$)
2) it is prone to instability ($J\lt 1$)

These are serious problems for a biological system, so this is not a satisfactory solution.

How can we address these issues? Let us consider a fundamental architectural motif in neural circuits in the cerebral cortex: neurons are either excitatory or inhibitory, and are strongly coupled with one another. 

# The E-I circuit

![](EInetwork.svg)

We thus consider two neurons, one excitatory, one inhibitory, mutually connected. The network model is now mathematically expressed with this system of coupled first-order ordinary differential equations:

$$
\tau \frac{d r_E(t)}{dt} = -r_E(t) + G_E \left[ W_{EE} r_E(t) - W_{EI} r_I(t) + I_E(t) - \theta_E \right]_{+}
$$

$$
\tau \frac{d r_I(t)}{dt} = -r_I(t) + G_I \left[ W_{IE} r_E(t) - W_{II} r_I(t) + I_I(t) - \theta_I \right]_{+} 
$$

where $W_{XY}$ are all non-negative and denote the strengths of the couplings between neuron $X$ and neuron $Y$. To ensure that firing rates are positive, we apply a linear-threshold transformation $G[\;I - \theta ]_{+}$ to the inputs, with thresholds $\theta_E$ and $\theta_I$. We again discretize this equation in time (in steps $dt$) using the Euler method to get:

$$
r_E(t+dt) = r_E(t) + \frac{dt}{\tau} \left( -r_E(t) + G_E \left[ W_{EE} r_E(t) - W_{EI} r_I(t) + I_E(t) - \theta_E \right]_+ \right)
$$

$$
r_I(t+dt) = r_I(t) + \frac{dt}{\tau} \left( -r_I(t) + G_I \left[ W_{IE} r_E(t) -W_{II} r_I(t) + I_I(t) - \theta_I \right]_+ \right)
$$

This system now is quite rich and can display a number of interesting dynamics. For instance, it can show the memory function that we discussed, but in a much more robust system. You can play with connectivity strengths in this code and convince you that this perfect memory is not subject to strict fine-tuning, thanks to the interaction of excitation and inhibition. For a detailed analysis of this system dynamics, you can explore [this article ](https://doi.org/10.7554/eLife.22425).

In [ ]:
dt = 0.0005
time = np.arange(0, 5, dt)
Nt = len(time)

input = np.zeros((Nt,))
pulsesat = int(Nt/6)
input[pulsesat:pulsesat+150] = 0.4

tauE = 0.01
tauI = 0.002
WEE = 5
WEI = 1
WIE = 10
WII = 0.5
GE = 1
GI = 4

rateI = np.zeros((Nt,))
rateE = np.zeros((Nt,))

def rectify(x, threshold):
    return (x - threshold) * (x > threshold)

for i in range(Nt-1):
    rateE[i+1] = rateE[i] + dt/tauE*( -rateE[i] + GE * rectify(WEE*rateE[i] - WEI*rateI[i] + input[i], 0.35) )
    rateI[i+1] = rateI[i] + dt/tauI*( -rateI[i] + GI * rectify(WIE*rateE[i] - WII*rateI[i] , 25) )

fig, ax = plt.subplots(nrows=2)
ax[0].plot(time, input, 'k')
ax[0].set_title('Input')
ax[0].set_ylabel('current')
ax[1].plot(time, rateE, 'r', label='E-cell')
ax[1].plot(time, rateI, 'b', label='I-cell')
ax[1].set_title('Output')
ax[1].set_xlabel('time (s)')
ax[1].set_ylabel('rate')
ax[1].legend()

### Advanced topic 1: Inhibition-stabilized network (skip if delayed)

How does this network respond to an excitatory input pulse applied to the excitatory population? and to the inhibitory population? 

In [ ]:
dt = 0.0005
time = np.arange(0, 5, dt)
Nt = len(time)

# initialize all inputs to zero
inputE = np.zeros((Nt,))
inputI = np.zeros((Nt,))

# first pulse to go into the non-zero solution
pulsesat = int(Nt/6)
inputE[pulsesat:pulsesat+150] = 0.4 

# now let's apply a pulse in the middle of the simulation to either the excitatory or the inhibitory neuron
# PLAY WITH THE TWO POSSIBLE INPUTS inputI AND inputE
pulsesat = int(3*Nt/6)
inputI[pulsesat:pulsesat+300] = 4
#inputE[pulsesat:pulsesat+300] = 4

tauE = 0.01
tauI = 0.002
WEE = 5
WEI = 1
WIE = 10
WII = 0.5
GE = 1
GI = 4

rateI = np.zeros((Nt,))
rateE = np.zeros((Nt,))

def rectify(x, threshold):
    return (x - threshold) * (x > threshold)

for i in range(Nt-1):
    rateE[i+1] = rateE[i] + dt/tauE*( -rateE[i] + GE * rectify(WEE*rateE[i] - WEI*rateI[i] + inputE[i], 0.35) )
    rateI[i+1] = rateI[i] + dt/tauI*( -rateI[i] + GI * rectify(WIE*rateE[i] - WII*rateI[i] + inputI[i], 25) )

fig, ax = plt.subplots(nrows=2)
ax[0].plot(time, inputE, 'r')
ax[0].plot(time, inputI, 'b')
ax[0].set_title('InputE')
ax[0].set_ylabel('current')
ax[1].plot(time, rateE, 'r', label='E-cell')
ax[1].plot(time, rateI, 'b', label='I-cell')
ax[1].set_title('Output')
ax[1].set_xlabel('time (s)')
ax[1].set_ylabel('rate')
ax[1].legend()

What do you see? Do you see a paradox? This network regime has been named "inhibition-stabilized network" or ISN. You can learn more about it in [this article](https://doi.org/10.1016/j.neuron.2009.03.028). This regime of operation is now considered to apply quite generally to neural circuits in the cerebral cortex (see [this article](https://doi.org/10.7554/eLife.54875)).

### Advanced topic 2: phase-plane analysis (skip if delayed)

In [ ]:
import brainpy as bp
import brainpy.math as bm

bm.enable_x64()
bm.set_platform('cpu')

@bp.odeint
def int_E(rE, t, rI, input=0.1, WEE=WEE, thrE=3, tauE=tauE, GE=GE):
    return - rE / tauE + GE * rectify(WEE*rE - WEI*rI + input, thrE) / tauE

@bp.odeint
def int_I(rI, t, rE, input=0.1, WIE=WIE, thrI=25, tauI=tauI, GI=GI):
    return - rI / tauI + GI * rectify(WIE*rE - WII*rI + input, thrI)  / tauI


analyzer = bp.analysis.PhasePlane2D(
    model=[int_E, int_I],
    target_vars={'rE': [0, 5], 'rI': [0, 20]},
    pars_update={'thrE': 1},
    # pars_update={'input': 0, 'WEE': 5, 'thrE': 3, 'tauE': 0.25, 'tauI': 0.01, 'GI': 4, 'thrI': 25},
    resolutions=0.05,
)
analyzer.plot_vector_field()
analyzer.plot_nullcline(coords=dict(rI='rE-rI'),
                        x_style={'fmt': '-'},
                        y_style={'fmt': '-'})
analyzer.plot_fixed_point()

plt.gca().set_box_aspect(1)
plt.ylabel('$rate_I$')
plt.xlabel('$rate_E$')
plt.tight_layout()

## EI network with adaptation (continue here)

This simple EI network can thus show robust bistability between a silent state and an active state. One interesting situation occurs when we consider the effect of one conspicuous neurobiological mechanism in cortical excitatory neurons: spike-frequency adaptation. 

In the cortex, when pyramidal cells are stimulated with constant current, they emit action potentials, but the frequency with which action potentials are emitted decreases with time: it adapts. This is due to the activation of calcium-dependent potassium currents in these cells. A simple way to describe the activation of these potassium currents (or adaptation currents, $I_A$) is through this equation: 
$$
\tau_A \frac{d I_A}{dt} = - I_A + g_A r_E
$$ 

Now go ahead and discretize this equation as we did above for other similar equations:

Then, excitatory neurons will experience this as an additional inhibitory input during the dynamics:

$$
\tau \frac{d r_E(t)}{dt} = -r_E(t) + G_E \left[ W_{EE} r_E(t) - W_{EI} r_I(t) + I_E(t) -I_A(t) - \theta_E \right]_{+}
$$

$$
\tau \frac{d r_I(t)}{dt} = -r_I(t) + G_I \left[ W_{IE} r_E(t) - W_{II} r_I(t) + I_I(t) - \theta_I \right]_{+} 
$$

In [ ]:
dt = 0.0005
time = np.arange(0, 10, dt)
Nt = len(time)

# initialize all inputs to zero
inputE = np.ones((Nt,))
inputI = np.ones((Nt,))

tauE = 0.01
tauI = 0.002
WEE = 5
WEI = 1
WIE = 10
WII = 0.5
GE = 1
GI = 4

tauA = 0.75
gA = 5

rateI = np.zeros((Nt,))
rateE = np.zeros((Nt,))
IA = np.zeros((Nt,))

def rectify(x, threshold):
    return (x - threshold) * (x > threshold)

for i in range(Nt-1):
    rateE[i+1] = rateE[i] + dt/tauE*( -rateE[i] + GE * rectify(WEE*rateE[i] - WEI*rateI[i] -IA[i] + inputE[i], 0.35) )
    rateI[i+1] = rateI[i] + dt/tauI*( -rateI[i] + GI * rectify(WIE*rateE[i] - WII*rateI[i] + inputI[i], 25) )
    IA[i+1] = IA[i] + dt/tauA * (-IA[i] + gA * rateE[i])

fig, ax = plt.subplots(nrows=2)
ax[0].plot(time, rateE, 'r')
ax[0].set_ylabel('rateE')
ax[1].plot(time, rateI, 'b')
ax[1].set_xlabel('time (s)')
ax[1].set_ylabel('rateI');

You should obtain here some interesting dynamics. What is the network doing? How would you describe this?

As an exercise, change the value of the adaptation current decay time constant, $\tau_A$. What does this parameter control in this dynamics? Can you conceptually understand why it does it?

This dynamics is a good model to analyze and interpret neural activity observed in neuronal cultures in vitro, in anesthetized animals in vivo, or in the deep phase of natural sleep. If you are interested in understanding better the potential of this model in these contexts, and especially for neuronal cultures you are in luck: Jaime next week will analyze this in more detail in his lectures, and you have the opportunity to choose a specific project to directly work on these models and their fit to specific data sets.

# Double-well attractor

![](doublewell.svg)

Now this starts getting complicated. Following the previous schematic for the EI network above, you could certainly write the code for this 3-neuron network (in practice, we think of each unit in the network as a population of neurons, so we talk of a 3-population network). This is an exercise for you to try on your own (you can get help from the presenation slides, too). Try this after class.

However, we usually simplify things to deal with only two dynamical variables. Here, we can take advantage of the fact that inhibitory neurons have a faster time constant to assume that their equation relaxes more quickly to the steady state, so we freeze the firing rate of the inhibitory population to the steady state of its dynamics given the momentary rates of the two excitatory populations. If you are interested in these derivations, you can check them in [this book chapter](https://neuronaldynamics.epfl.ch/online/Ch16.S3.html). The resulting equations are:

$$
\tau_E \frac{d I_{E,1}}{dt} = - I_{E1} + (W_{EE} - \alpha) g_E(I_{E1}) - \alpha g_E(I_{E2}) + S_{1}
$$

$$
\tau_E \frac{d I_{E,2}}{dt} = - I_{E2} + (W_{EE} - \alpha) g_E(I_{E2}) - \alpha g_E(I_{E1}) + S_{2} 
$$

where $\alpha = -\gamma W_{EI}W_{IE}$ represents the effective inhibitory coupling between excitatory populations (via the inhibitory population), and $S_1$/$S_2$ are the inputs arriving to each excitatory population. 

 We can simplify the parametrization of these equations by defining $J_E=W_{EE}-\alpha$ and $J_I=\alpha$ to get the [Wong and Wang model](https://doi.org/10.1523/JNEUROSCI.3733-05.2006):


$$
\tau_E \frac{d I_{E,1}}{dt} = - I_{E1} + J_E g_E(I_{E1}) - J_I g_E(I_{E2}) + S_{1}
$$

$$
\tau_E \frac{d I_{E,2}}{dt} = - I_{E2} + J_E g_E(I_{E2}) - J_I g_E(I_{E1}) + S_{2} 
$$


Note that this model formulation is slightly different to what we have been doing so far in this workshop: now our dynamical variables are the inputs and not the rates of the populations. To plot the rates, we have to use $r_{E} = g_E(I_E)$.

These equations can be discretized to obtain this simulation code:

In [ ]:
dt = 0.0005
time = np.arange(0, 10, dt)
Nt = len(time)

# initialize inputs 
drive0 = 0.05
inputE1 = drive0*np.ones((Nt,))
inputE2 = drive0*np.ones((Nt,))


# current pulse into one of the two populations
pulsesat = int(Nt/6)
inputE1[pulsesat:pulsesat+150] = 0.4 


tauE = 0.05
JE = 1.1
JI = 1.8

inpE1 = -0.005*np.ones((Nt,))
inpE2 = -0.005*np.ones((Nt,))

def curr_to_rate(x):
    return (1+np.tanh(x-0.5))/2

for i in range(Nt-1):
    inpE1[i+1] = inpE1[i] + dt/tauE*(-inpE1[i] + JE*curr_to_rate(inpE1[i]) - JI*curr_to_rate(inpE2[i]) + inputE1[i] )
    inpE2[i+1] = inpE2[i] + dt/tauE*(-inpE2[i] + JE*curr_to_rate(inpE2[i]) - JI*curr_to_rate(inpE1[i]) + inputE2[i] )

rateE1 = curr_to_rate(inpE1)
rateE2 = curr_to_rate(inpE2)

fig, ax = plt.subplots(nrows=2)
ax[0].plot(time, inputE1, 'r')
ax[0].plot(time, inputE2, 'm')
ax[0].set_title('InputE')
ax[0].set_ylabel('current')
ax[1].plot(time, rateE1, 'r', label='Population 1')
ax[1].plot(time, rateE2, 'm', label='Population 2')
ax[1].set_title('Output')
ax[1].set_xlabel('time (s)')
ax[1].set_ylabel('rate')
ax[1].legend();

What happens if you send the input current pulse to population 2 instead of population 1? Argue about how this model can be used to remember specific events that happened in the recent past. 

These discrete attractor models have been applied to model working memory for objects. In this case there are just two possible memories, because there are two populations, but this can be generalized to an arbitrary number of discrete memories. If you want to learn more about these models, you can check [this paper](https://www.nature.com/articles/s41586-019-0919-7) on experimental evidence supporting it.

Also, notice how this network now responds to input by establishing a competition between the two populations: if one wins, the other one loses. This is what we call "winner-take-all" dynamics and it is thought also to be a mechanism underlying multiple brain computations from sensory perception to decision making.

The advantage of having just two dynamical variables is that we can visualize the dynamics in what we call "phase space":

In [ ]:
dt = 0.0005
time = np.arange(0, 10, dt)
Nt = len(time)

# initialize inputs 
drive0 = 0.
inputE1 = drive0*np.ones((Nt,))
inputE2 = drive0*np.ones((Nt,))

drive = 0.1
inputE1[Nt//4:] = drive
inputE2[Nt//4:] = drive

# first pulse to go into the non-zero solution
pulsesat = int(Nt/6)
inputE2[pulsesat:pulsesat+150] = 0.4 


tauE = 0.05
JE = 1.1
JI = 1.8

inpE1 = -0.005*np.ones((Nt,))
inpE2 = -0.005*np.ones((Nt,))

def curr_to_rate(x):
    return (1+np.tanh(x-0.5))/2

sigma = 0.3

for i in range(Nt-1):
    noise = sigma * np.random.randn()
    inpE1[i+1] = inpE1[i] + dt/tauE*(-inpE1[i] + JE*curr_to_rate(inpE1[i]) - JI*curr_to_rate(inpE2[i]) + inputE1[i])
    noise = sigma * np.random.randn()
    inpE2[i+1] = inpE2[i] + dt/tauE*(-inpE2[i] + JE*curr_to_rate(inpE2[i]) - JI*curr_to_rate(inpE1[i]) + inputE2[i])

rateE1 = curr_to_rate(inpE1)
rateE2 = curr_to_rate(inpE2)

plt.scatter(rateE1, rateE2, c=time, s=4, cmap='cool')
plt.colorbar(label='time')
plt.xlim([0,1])
plt.ylim([0,1])
plt.axline((1, 1), slope=1, color='k', ls='--')
plt.xlabel('rate population 2')
plt.ylabel('rate population 1');

try running several times the previous cell, changing the transient input to the E1 or E2 populations, and see how the network dynamics changes from trial to trial. Can you make sense of this dynamics? Where does the network start and where does it evolve to towards the end of the trial?

One very useful way to think about this dynamics is to visualize it as the evolution of a ball that bounces down in a hilly landscape. This is more than just a visual analogy for networks that satisfy perfect symmetry (i.e. the connection from neuron X to neuron Y is equal to the connection from neuron Y to neuron X). For these kinds of networks this analogy is mathematically exact (see for example an explanation [here](https://neuronaldynamics.epfl.ch/online/Ch16.S4.html)) and this hilly landscape is what we call "energy". See here the energy calculated for the network above:

In [ ]:
np.seterr(divide = 'ignore') 

energy = np.zeros((100,100))

def integral(rate):
    lim = 2*rate - 1
    return 0.5*rate + 0.5* (lim*np.arctanh(lim) + 0.5*np.log(np.abs(1-lim*lim)) -0.7);

for i in range(100):
    r1 = i/100
    for j in range(100):
        r2 = j/100
        energy[i,j] = -0.5*JE*(r1**2 + r2**2) + JI*r1*r2 - (drive*r1 + drive*r2) + integral(r1) + integral(r2)

plt.contour(energy, extent=[0, 1, 0, 1], levels=500)
plt.colorbar(label='energy')
plt.xlabel('rate population 2')
plt.ylabel('rate population 1')


plt.scatter(rateE1, rateE2, c=time, s=4, cmap="cool")
plt.colorbar(label='time')
plt.xlim([0,1])
plt.ylim([0,1])
plt.axline((1, 1), slope=1, color='k', ls='--')
plt.xlabel('rate population 2')
plt.ylabel('rate population 1');

As you can see, the trajectories in phase space that we saw before are falling down this landscape towards the two "wells" of minimal energy. Based on this energy picture, we often call this dynamics "double-well attractor", and it is a possible circuit mechanism both for working memory and decision making (below).

Here is a more graphic view of the double well, when taking a cut through the previous 2D energy landscape following the line $r_2 + r_1 = 0.6$:

In [ ]:
r1 = np.linspace(0,0.6,100)
r2 = -r1+0.6

i1 = r1*100
i1 = i1.astype(int)
i2 = r2*100
i2 = i2.astype(int)

plt.plot(r2, energy[i1, i2]);
plt.xlabel('rate population 2')
plt.ylabel('energy');

### The double-well model in decision making

Now let's explore this network in a different situation: we do not establish a difference between the external inputs to the two populations but we add random independent noise to the currents of the two populations at each time step in the simulated dynamics. This represents internal noise of the brain. Explore what happens in this simulation by running it repeatedly over several "trials". Notice also that we change the overall drive to the network during the simulation, but keeping it equal for the two populations. What does this change in external drive achieve?

In [ ]:
dt = 0.0005
time = np.arange(0, 10, dt)
Nt = len(time)

# initialize inputs 
drive0 = 0.
inputE1 = drive0*np.ones((Nt,))
inputE2 = drive0*np.ones((Nt,))

drive = 0.1
inputE1[Nt//4:] = drive
inputE2[Nt//4:] = drive

# first pulse to go into the non-zero solution
pulsesat = int(Nt/6)
#inputE1[pulsesat:pulsesat+150] = 0.4 


tauE = 0.05
JE = 1.1
JI = 1.8

inpE1 = -0.005*np.ones((Nt,))
inpE2 = -0.005*np.ones((Nt,))

def curr_to_rate(x):
    return (1+np.tanh(x-0.5))/2

sigma = 0.3

for i in range(Nt-1):
    noise = sigma * np.random.randn()
    inpE1[i+1] = inpE1[i] + dt/tauE*(-inpE1[i] + JE*curr_to_rate(inpE1[i]) - JI*curr_to_rate(inpE2[i]) + inputE1[i] + noise)
    noise = sigma * np.random.randn()
    inpE2[i+1] = inpE2[i] + dt/tauE*(-inpE2[i] + JE*curr_to_rate(inpE2[i]) - JI*curr_to_rate(inpE1[i]) + inputE2[i] + noise)

rateE1 = curr_to_rate(inpE1)
rateE2 = curr_to_rate(inpE2)

fig, ax = plt.subplots(nrows=2)
ax[0].plot(time, inputE1, 'r')
ax[0].plot(time, inputE2, 'm')
ax[0].set_title('InputE')
ax[0].set_ylabel('current')
ax[1].plot(time, rateE1, 'r', label='Population 1')
ax[1].plot(time, rateE2, 'm', label='Population 2')
ax[1].set_title('Output')
ax[1].set_xlabel('time (s)')
ax[1].set_ylabel('rate')
ax[1].legend();

Which of the two populations wins now the competition? How is that determined?

Explore now the dynamics of the model in phase space, for different realizations of the noise:

In [ ]:
dt = 0.0005
time = np.arange(0, 10, dt)
Nt = len(time)

# initialize inputs 
drive0 = 0.
inputE1 = drive0*np.ones((Nt,))
inputE2 = drive0*np.ones((Nt,))

drive = 0.1
inputE1[Nt//4:] = drive
inputE2[Nt//4:] = drive

inpE1 = -0.005*np.ones((Nt,))
inpE2 = -0.005*np.ones((Nt,))

def curr_to_rate(x):
    return (1+np.tanh(x-0.5))/2

sigma = 0.3

for i in range(Nt-1):
    noise = sigma * np.random.randn()
    inpE1[i+1] = inpE1[i] + dt/tauE*(-inpE1[i] + JE*curr_to_rate(inpE1[i]) - JI*curr_to_rate(inpE2[i]) + inputE1[i] + noise)
    noise = sigma * np.random.randn()
    inpE2[i+1] = inpE2[i] + dt/tauE*(-inpE2[i] + JE*curr_to_rate(inpE2[i]) - JI*curr_to_rate(inpE1[i]) + inputE2[i] + noise)

rateE1 = curr_to_rate(inpE1)
rateE2 = curr_to_rate(inpE2)

np.seterr(divide = 'ignore') 

energy = np.zeros((100,100))

def integral(rate):
    lim = 2*rate - 1
    return 0.5*rate + 0.5* (lim*np.arctanh(lim) + 0.5*np.log(np.abs(1-lim*lim)) -0.7);

for i in range(100):
    r1 = i/100
    for j in range(100):
        r2 = j/100
        energy[i,j] = -0.5*JE*(r1**2 + r2**2) + JI*r1*r2 - (drive*r1 + drive*r2) + integral(r1) + integral(r2)

plt.contour(energy, extent=[0, 1, 0, 1], levels=500)
plt.colorbar(label='energy')
plt.xlabel('rate population 2')
plt.ylabel('rate population 1')


plt.scatter(rateE1, rateE2, c=time, s=4, cmap="cool")
plt.colorbar(label='time')
plt.xlim([0,1])
plt.ylim([0,1])
plt.axline((1, 1), slope=1, color='k', ls='--')
plt.xlabel('rate population 2')
plt.ylabel('rate population 1');

Now, remember that in our dynamical simulation we changed the external drive to the simulation and that started the decision process of the network. As an exercise, visualize the energy landscape for the external drive condition at the start of the simulation. Does it make sense to you? 

In [ ]:
np.seterr(divide = 'ignore') 

energy = np.zeros((100,100))

def integral(rate):
    lim = 2*rate - 1
    return 0.5*rate + 0.5* (lim*np.arctanh(lim) + 0.5*np.log(np.abs(1-lim*lim)) -0.7)

for i in range(100):
    r1 = i/100
    for j in range(100):
        r2 = j/100
        energy[i,j] = -0.5*JE*(r1**2 + r2**2) + JI*r1*r2 - (drive0*r1 + drive0*r2) + integral(r1) + integral(r2)

plt.contour(energy, extent=[0, 1, 0, 1], levels=500)
plt.colorbar(label='energy')
plt.xlabel('rate population 2')
plt.ylabel('rate population 1');

In [ ]:
r1 = np.linspace(0,0.45,100)
r2 = -r1+0.45

i1 = r1*100
i1 = i1.astype(int)
i2 = r2*100
i2 = i2.astype(int)

plt.plot(r2, energy[i1, i2]);
plt.xlabel('rate population 2')
plt.ylabel('energy');

The energy landscape can also help us understand what happens in the model when we imbalance the inputs to the two populations. For instance, if the drive to population 2 is stronger than the drive to population 1, simulating the case in which stronger evidence is presented for stimulus 2 than for stimulus 1. This biases the competition between the two populations in favor of population 2 by making the well of population 1 much shallower, or even making it disappear:

In [ ]:
energy = np.zeros((100,100))
drive1 = 0.08
drive2 = 0.12

def integral(rate):
    lim = 2*rate - 1
    return 0.5*rate + 0.5* (lim*np.arctanh(lim) + 0.5*np.log(np.abs(1-lim*lim)) -0.7)

for i in range(100):
    r1 = i/100
    for j in range(100):
        r2 = j/100
        energy[i,j] = -0.5*JE*(r1**2 + r2**2) + JI*r1*r2 - (drive1*r1 + drive2*r2) + integral(r1) + integral(r2)

plt.contour(energy, extent=[0, 1, 0, 1], levels=500)
plt.colorbar(label='energy')
plt.xlabel('rate population 2')
plt.ylabel('rate population 1');

In [ ]:
r1 = np.linspace(0,0.5,100)
r2 = -r1+0.5

i1 = r1*100
i1 = i1.astype(int)
i2 = r2*100
i2 = i2.astype(int)

plt.plot(r2, energy[i1, i2]);
plt.xlabel('rate population 2')
plt.ylabel('energy')

If you want to learn more about how this model has been applied to decision making, you are in luck: you can explore it in depth in one of the proposed projects. Also, you can read [this paper](https://www.jneurosci.org/content/26/4/1314.short) by Wong and Wang.

### Advanced topic 3: phase plane analysis in the double-well model (skip if delayed)

Try the two parameter sets used above to get 1) a double-well dynamics, and 2) a single-well dynamics in the cell code below. Can you make sense of this dynamics from the intersections of the nullclines?

In [ ]:
drive = drive0

@bp.odeint
def int_r1(r1, t, r2, drive=drive):
    fct = 2*r1*(1.-r1)/tauE
    cnv = 0.5 + bm.atanh(2*r1 - 1.)
    return (- cnv + JE*r1 - JI*r2 + drive) *fct

@bp.odeint
def int_r2(r2, t, r1, drive=drive):
    fct = 2*r2*(1.-r2)/tauE
    cnv = 0.5 + bm.atanh(2*r2 - 1.)
    return (- cnv + JE*r2 - JI*r1 + drive) *fct


analyzer = bp.analysis.PhasePlane2D(
    model=[int_r1, int_r2],
    target_vars={'r1': [0, 1], 'r2': [0, 1]},
    # pars_update={'drive': -0.5},
    resolutions=0.0005,
)
analyzer.plot_vector_field()
analyzer.plot_nullcline(coords=dict(r2='r2-r1'),
                        x_style={'fmt': '-'},
                        y_style={'fmt': '-'})
analyzer.plot_fixed_point()
plt.gca().set_box_aspect(1)
plt.tight_layout()

You can now compute a bifurcation diagram where you track the location of the intersection of the nullclines (fixed points) as you gradually change one control parameter (for instane the external *drive*). Can you now understand how the system goes from a single well to a double well by changing the external drive?

In [ ]:
@bp.odeint
def int_s1(s1, t, s2, drive=drive):
    crE1 = (1 + bm.tanh(s1 - 0.5))/2.
    crE2 = (1 + bm.tanh(s2 - 0.5))/2.
    return - s1 / tauE + JE*crE1 / tauE - JI*crE2 / tauE + drive / tauE

@bp.odeint
def int_s2(s2, t, s1, drive=drive):
    crE1 = (1 + bm.tanh(s1 - 0.5))/2.
    crE2 = (1 + bm.tanh(s2 - 0.5))/2.
    return - s2 / tauE + JE*crE2 / tauE - JI*crE1 / tauE + drive / tauE

analyzer = bp.analysis.Bifurcation2D(
  model=[int_s1, int_s2],
  target_vars={'s1': [-1.5, 1.5], 's2': [-1.5, 1.5]},
  target_pars={'drive': [-0.2, 0.5]},
  resolutions={'drive': 0.01},
)

analyzer.plot_bifurcation(num_rank=50)

plt.close();
plt.gca().set_box_aspect(1)
plt.xlabel('drive')
plt.ylabel('current E1 (s1)')
plt.tight_layout()

as an exercise, try to get the bifurcation diagram when the parameter $J_E$ is changed gradually

If you still feel lucky, try going from this "supercritical bifurcation" to a "subcritical bifurcation", where there is a region of bistability between three stable fixed points. Can you find that?